In [5]:
import pandas as pd
import mhcgnomes
import uuid
pd.set_option("display.max_columns",500)
PROTEIN_CHECK="^[ACDEFGHIKLMNPQRSTVWY]+$"
HOST_SPECIES = ['human', 'mouse']
CDR3_CHAINS = ['alpha', 'beta']

In [16]:
def fix_mhc_name(allele_str, chain = 'alpha'):
    try:
        # Предварительные проверки и подготовления
        if pd.isna(allele_str):
            return pd.NA     
        allele_first = allele_str.split(" ")[0]
        if allele_first == "B2M":
            return "B2M"
        # Получение конкретной аллели
        parsed_allele = mhcgnomes.parse(allele_first)
        if isinstance(parsed_allele, mhcgnomes.pair.Pair):
            if chain == 'alpha':
                allele = parsed_allele.alpha
            elif chain == 'beta':
                allele = parsed_allele.beta
            else:
                raise ValueError('Unknown chain')
        elif isinstance(parsed_allele, mhcgnomes.allele.Allele):
            allele = parsed_allele
        else:
            raise ValueError('Not allele')
        # Проверка
        if allele.gene.species.name == "Homo sapiens":
            if len(allele.allele_fields) < 2:
                raise ValueError('Too many allele fields. Need at least 2.')
            elif len(allele.allele_fields) == 2:
                return allele.to_string()
            else:
                return allele.restrict_allele_fields(2, drop_annotations=True, drop_mutations=True).to_string()
        elif allele.gene.species.name == "Mus musculus":
            return allele.to_string()
        else:
            raise ValueError('Wrong species. Only human and mouse are allowed.')
    except (mhcgnomes.errors.ParseError, TypeError, ValueError):
        return pd.NA
    except AttributeError:
        print(allele_first)
        return pd.NA

In [17]:
raw_data = pd.read_csv("../../data/raw-data/PIRD/PIRD.csv",sep = ';',na_values = "-")
raw_data.head()

/scratch/ipykernel_3383778/2155345946.py:1: DtypeWarning: Columns (10,24) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_data = pd.read_csv("../../data/raw-data/PIRD/PIRD.csv",sep = ';',na_values = "-")


,ICDname,Disease.name,Category,Antigen,Antigen.sequence,HLA,Locus,CDR3.alpha.aa,CDR3.beta.aa,CDR3.alpha.nt,CDR3.beta.nt,Valpha,Jalpha,Vbeta,Dbeta,Jbeta,Seq.platform,Species,Origin,Nucleotide.type,Cell.subtype,Prepare.method,Evaluate.method,Case.num,Control.type,Control.num,Filteration,Journal,Pubmed.id,Grade
0,A15,Tuberculosis,Pathogen,CFP10,TAAQAAVVRFQEAAN,DRB1*15:03,TRA-TRB,CIEHTNSGGSNYKLTF,CASSLEETQYF,NaN,NaN,NaN,NaN,NaN,NaN,TRBJ2-5,NGS,Homo Sapiens,PBMC,RNA,CD4,Multiple PCR,Antigen-specific ex vivo proliferation,22.0,NaN,NaN,NaN,Nature,28636589,5
1,A15,Tuberculosis,Pathogen,CFP10,TAAQAAVVRFQEAAN,DRB1*15:03,TRA-TRB,CIVHTNSGGSNYKLTF,CASSPEETQYF,NaN,NaN,NaN,NaN,NaN,NaN,TRBJ2-5,NGS,Homo Sapiens,PBMC,RNA,CD4,Multiple PCR,Antigen-specific ex vivo proliferation,22.0,NaN,NaN,NaN,Nature,28636589,5
2,A15,Tuberculosis,Pathogen,CFP10,TAAQAAVVRFQEAAN,DRB1*15:03,TRA-TRB,CIVKTNSGGSNYKLTF,CASSFEETQYF,NaN,NaN,NaN,NaN,NaN,NaN,TRBJ2-5,NGS,Homo Sapiens,PBMC,RNA,CD4,Multiple PCR,Antigen-specific ex vivo proliferation,22.0,NaN,NaN,NaN,Nature,28636589,5
3,A15,Tuberculosis,Pathogen,ESAT-6;CFP-10,NaN,NaN,TRB,NaN,CASGRPYEQYF,NaN,NaN,NaN,NaN,TRBV13-1,NaN,NaN,ABI 3730,Homo Sapiens,PBMC,RNA,T,5'RACE,Statistical analysis,25.0,Health,15.0,Stimulated with ESAT-6 or CFP-10,Tuberculosis,23845455,2
4,A15,Tuberculosis,Pathogen,ESAT-6;CFP-10,NaN,NaN,TRB,NaN,CASSFLERGLFFYEQYF,NaN,NaN,NaN,NaN,TRBV3,NaN,NaN,ABI 3730,Homo Sapiens,PBMC,RNA,T,5'RACE,Statistical analysis,25.0,NaN,15.0,Stimulated with ESAT-6 or CFP-10,Tuberculosis,23845455,2


In [18]:
raw_data['HLA'].unique()

array(['DRB1*15:03', nan, 'DRB1*15:01/DRB5*01:01', 'A*02:01', 'B*07:02',
       'A*01:01', 'B*08:01',
       'A*02-A*11/B*38-B*67/C*01-C*07/DR9-DRw3/DQw3-DQw5',
       'A*24-A*26/B*61/Cw*03/DRB1*09-DRB1*01/DQB1*0306', 'HLAI',
       'B*51:01', 'B*57', 'B*57:01', 'B*57:03', 'B*27:05', 'B*27', 'B*42',
       'B*08', 'B*15', 'B*42:01', 'A*02-A*26/B*07-B38/Bw*06/Cw*04',
       'A*02-A*30', 'A*02/B*39-B44/Bw*06-Bw*04/Cw*05-Cw*07',
       'A*01:01-A*02:01/B*08:01-B*57:01/Cw*06:02-Cw*07:01/DRB1*08:0321-DRB1*15:011',
       'A*24:02', 'A-B*18', 'A-B*08:01', 'A-B*08', 'A-A*02:01', 'B*35:08',
       'B*35:01', 'A*02', 'B*07', 'A-A*01:01',
       'A*01-A*02/B*35-B*44/Cw*04-Cw*05/DRB1*07-DRB1*1204/DQB1*03',
       'A*01-A*11/B*08-B*56/Cw*01-Cw*07/DRB1*03-DRB1*04/DQB1*02-DQB1*03 ',
       'A*01-A*11/B*08-B*56/Cw01-Cw*07/DRB1*03-DRB1*04/DQB1*02-DRB1*03 ',
       'A*01-A*32/B*07-B*62/Cw*04-Cw*07/DRB1*08-DRB1*15/DQB1*04-DQB1*06',
       'A*26', 'A*02-A*19/B*44-B*51',
       'Complete HLA typing is pro

In [14]:
mhcgnomes.parse('A*02:01')

Allele(gene=Gene(species=Species(name='Homo sapiens', mhc_prefix='HLA'), name='A', mutations=()), allele_fields=('02', '01'), annotations=(), mutations=())

In [19]:
print(raw_data.shape)
raw_data['host_species'] = 'human'
raw_data['id'] = raw_data['ICDname'].apply(lambda _: str(uuid.uuid4()))
raw_data['HLA'] = "HLA-" + raw_data['HLA'] 
raw_data['valid_ii_name'] = "B2M" # not valid mhcii, they will be filtered
raw_data['valid_i_name'] = raw_data['HLA'].apply(lambda x: fix_mhc_name(x, 'alpha'))
raw_data["database"] = "PIRD"
raw_data.head()

(51139, 30)
HLA-A-B*18
HLA-A-B*18
HLA-A-B*18
HLA-A-B*18
HLA-A-B*18
HLA-A-B*18
HLA-A-B*18
HLA-A-B*18
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08
HLA-A-B*08
HLA-A-B*08
HLA-A-B*08
HLA-A-A*02:01
HLA-A-A*02:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08
HLA-A-B*08
HLA-A-B*08
HLA-A-B*08
HLA-A-B*08
HLA-A-B*08
HLA-A-B*08
HLA-A-B*08
HLA-A-B*08
HLA-A-B*08
HLA-A-B*08
HLA-A-B*08
HLA-A-B*08
HLA-A-B*08
HLA-A-B*08
HLA-A-B*08
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-B*08:01
HLA-A-A*02:01
HLA-A-A*02:01
HLA-A-A*02:01
HLA-A-A*02:01
HLA-A-A*

,ICDname,Disease.name,Category,Antigen,Antigen.sequence,HLA,Locus,CDR3.alpha.aa,CDR3.beta.aa,CDR3.alpha.nt,CDR3.beta.nt,Valpha,Jalpha,Vbeta,Dbeta,Jbeta,Seq.platform,Species,Origin,Nucleotide.type,Cell.subtype,Prepare.method,Evaluate.method,Case.num,Control.type,Control.num,Filteration,Journal,Pubmed.id,Grade,host_species,id,valid_ii_name,valid_i_name,database
0,A15,Tuberculosis,Pathogen,CFP10,TAAQAAVVRFQEAAN,HLA-DRB1*15:03,TRA-TRB,CIEHTNSGGSNYKLTF,CASSLEETQYF,NaN,NaN,NaN,NaN,NaN,NaN,TRBJ2-5,NGS,Homo Sapiens,PBMC,RNA,CD4,Multiple PCR,Antigen-specific ex vivo proliferation,22.0,NaN,NaN,NaN,Nature,28636589,5,human,979a0eb2-c195-4c34-9cae-a2e310fd17c6,B2M,HLA-DRB1*15:03,PIRD
1,A15,Tuberculosis,Pathogen,CFP10,TAAQAAVVRFQEAAN,HLA-DRB1*15:03,TRA-TRB,CIVHTNSGGSNYKLTF,CASSPEETQYF,NaN,NaN,NaN,NaN,NaN,NaN,TRBJ2-5,NGS,Homo Sapiens,PBMC,RNA,CD4,Multiple PCR,Antigen-specific ex vivo proliferation,22.0,NaN,NaN,NaN,Nature,28636589,5,human,3879a9b7-253f-41c7-8cb9-0f5d8585fe57,B2M,HLA-DRB1*15:03,PIRD
2,A15,Tuberculosis,Pathogen,CFP10,TAAQAAVVRFQEAAN,HLA-DRB1*15:03,TRA-TRB,CIVKTNSGGSNYKLTF,CASSFEETQYF,NaN,NaN,NaN,NaN,NaN,NaN,TRBJ2-5,NGS,Homo Sapiens,PBMC,RNA,CD4,Multiple PCR,Antigen-specific ex vivo proliferation,22.0,NaN,NaN,NaN,Nature,28636589,5,human,06eb8741-e24f-4a75-a3e3-ebd664bf44a8,B2M,HLA-DRB1*15:03,PIRD
3,A15,Tuberculosis,Pathogen,ESAT-6;CFP-10,NaN,NaN,TRB,NaN,CASGRPYEQYF,NaN,NaN,NaN,NaN,TRBV13-1,NaN,NaN,ABI 3730,Homo Sapiens,PBMC,RNA,T,5'RACE,Statistical analysis,25.0,Health,15.0,Stimulated with ESAT-6 or CFP-10,Tuberculosis,23845455,2,human,6123095f-5e13-4807-bdf9-6ba928bb2460,B2M,<NA>,PIRD
4,A15,Tuberculosis,Pathogen,ESAT-6;CFP-10,NaN,NaN,TRB,NaN,CASSFLERGLFFYEQYF,NaN,NaN,NaN,NaN,TRBV3,NaN,NaN,ABI 3730,Homo Sapiens,PBMC,RNA,T,5'RACE,Statistical analysis,25.0,NaN,15.0,Stimulated with ESAT-6 or CFP-10,Tuberculosis,23845455,2,human,56dba86a-25b7-4a14-bc92-b95ef314b43a,B2M,<NA>,PIRD


In [21]:
table_schema = {
            "id": "id",
            "CDR3.alpha.aa": "cdr3_alpha",
            "CDR3.beta.aa": "cdr3_beta",
            "Antigen.sequence": "epitope",
            "valid_i_name": "mhc_alpha",
            "valid_ii_name": "mhc_beta",
            "mhc_class": "mhc_class",
            "host_species": "host_species",
            "Disease.name": "epitope_species",
            "Antigen": "epitope_source",
            "Valpha": "V_alpha",
            "Vbeta": "V_beta",
            "Vbeta": "D_beta",
            "Jbeta": "J_alpha",
            "Jbeta": "J_beta", 
            "database": "database"
        }

In [26]:
print('Apply filters')
clean_data = raw_data.\
          query("`Pubmed.id`.notna()").\
          query("`CDR3.alpha.aa`.fillna('').str.contains(@PROTEIN_CHECK) or `CDR3.beta.aa`.fillna('').str.contains(@PROTEIN_CHECK)").\
          query("`Antigen.sequence`.notna()").\
          query("`Antigen.sequence`.str.contains(@PROTEIN_CHECK)").\
          query("valid_i_name.notna() and valid_ii_name.notna()")
print(clean_data.shape)

Apply filters
(2050, 35)


In [27]:
clean_data['mhc_class'] = clean_data['valid_i_name'].apply(lambda x: 'I' if mhcgnomes.parse(x).is_class1 else 'II')
clean_data = clean_data.query("mhc_class == 'I'") # no valid mhcII entries

In [29]:
clean_data["D_alpha"] = pd.NA

In [31]:
clean_data_selected = clean_data.filter(items = list(table_schema.keys()), axis = 1).rename(columns = table_schema)
clean_data_selected.shape

(2002, 14)